In [3]:
from fourier_xor_shuffle_2023_07_24 import train, replace_xors_with_random, shuffle_xors, identity
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from itertools import product
from pathlib import Path
from loguru import logger
from nn_xors_2023_07_18 import MLPBinaryClassifier, make_dataset
from spin_lattices import TriangleLattice, SquareLattice, KagomeLattice
from heisenberg_hamiltonians import HeisenbergJ1J2
import numpy as np
from lattice_boolean_analysis import LBFFromSpinSystem
from fast_boolean_analysis import FourierSeries, fourier_expand, keep_largest_n
from collections.abc import Callable
import fire
import json

self_name = "fourier_xor_shuffle_2023_07_25_1_ipynb"
# self_name = Path(__file__).stem
# output_dir = Path("experiments") / self_name
# output_dir.mkdir(exist_ok=True)

eps_train = 0.1
eps_test = 0.1

batch_size = 256
n_hidden = 512
epochs = 300
target_rel_weight = 0.2
runs = 20
break_on_loss = 1e-2
lr = 1e-3


def truncate(series: FourierSeries) -> FourierSeries:
    return series.truncate(
        keep_largest_n(series.how_many_terms_to_achieve_relative_weight(target_rel_weight))
    )


class Compose:
    def __init__(self, *funcs):
        self.funcs = funcs

    def __call__(self, x):
        for f in self.funcs:
            x = f(x)
        return x

    def __repr__(self):
        return "∘".join(f.__name__ for f in self.funcs[::-1])


system_specs = [
#    (TriangleLattice(6, 4), 1.3),
    (KagomeLattice(2, 4), 1.0),
]

transformations = [
    Compose(identity),
    Compose(truncate),
    Compose(truncate, replace_xors_with_random),
    Compose(truncate, shuffle_xors),
]


def main(task_id: int | None = None, runs: int = 20):
    task_list = list(product(system_specs, transformations))
    if task_id is None:
        print("You have to specify task_id. Available tasks:")
        for i, task in enumerate(task_list):
            print(i, task)
        return

    (lattice, J2), transform = task_list[task_id]
    system = HeisenbergJ1J2(lattice, J1=1, J2=J2, ground_state_cache_dir=Path("groundstates"))
    system.get_eigenstates(1)

    for run in range(runs):
        task_name = f"{system.get_cache_id()}_{transform}_{run}"
        n_spins = system.number_spins

        writer = SummaryWriter(
            log_dir=(
                f"experiments/{self_name}.tb/{datetime.now().strftime('%H_%M_%S')}" f"_{task_name}"
            )
        )

        all_states = system.canonical_basis.states
        sample_states = np.random.choice(
            all_states,
            size=int(len(all_states) * (eps_train + eps_test)),
            replace=False,
            p=system.get_ground_state_in_canonical_basis().astype(np.float64) ** 2,
        )

        signal = LBFFromSpinSystem(system)
        series = fourier_expand(signal)

        transformed_series = transform(series)
        dataset = make_dataset(transformed_series, sample_states, n_spins)

        train_dataset, test_dataset = random_split(
            dataset, [eps_train / (eps_train + eps_test), eps_test / (eps_train + eps_test)]
        )
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        net = MLPBinaryClassifier(n_spins, n_hidden)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(net.parameters(), lr=lr)

        output = train(
            net=net,
            criterion=criterion,
            optimizer=optimizer,
            train_loader=train_loader,
            test_dataset=test_dataset,
            n_epochs=epochs,
            writer=writer,
            break_on_loss=break_on_loss,
        )

        # Path(output_dir / f"{task_name}.json").write_text(
        #     json.dumps(
        #         output
        #         | {
        #             "transform": repr(transform),
        #             "run": run,
        #             "non_zero_coeffs": int((transformed_series.coeffs != 0).sum()),
        #             "system": system.get_cache_id(),
        #             "lattice": lattice.get_cache_id(),
        #         }
        #     )
        # )

In [4]:
main(0)

2023-07-25 18:32:16.971 | DEBUG    | heisenberg_hamiltonians:__init__:438 - use_symmetries is None and lattice is in symmetries whitelist, setting use_symmetries=True, spin_inversion=1
2023-07-25 18:32:16.985 | DEBUG    | heisenberg_hamiltonians:__init__:459 - number_spins=24
2023-07-25 18:32:16.995 | DEBUG    | heisenberg_hamiltonians:__init__:469 - Symmetry group contains 16 elements
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-25 18:32:17.091 | DEBUG    | heisenberg_hamiltonians:__init__:478 - Hilbert space dimension is 85662
2023-07-25 18:32:17.110 | DEBUG    | heisenberg_hamiltonians:_find_cached_eigenstate:67 - Using cached version of eigenvalues / eigenstates from groundstates/HeisenbergJ1J2-KagomeLattice2x4-1.0-1.0-True-1-1.pickle
2023-07-25 18:32:17.120 | DEBUG    | heisenberg_hamiltonians:get_eigenstates:114 - Ground state energy is -42.8245991763
[Debug]   [LOCALE0]   ls_chpl_enumerate_representatives ...
2023-07-25 18:32:17.229 | DEBUG    | heisenberg

KeyboardInterrupt: 